# Lesson 9: Parallel Arithmetic Execution

## Objective
Learn how to execute multiple arithmetic operations in parallel instead of sequentially using LangGraph's fan-out/fan-in pattern.

## Problem Statement
**Previous lesson (Lesson 8):** Tools executed one at a time in a loop.
- "Add 5 and 3, multiply 2 and 4" would:
  1. Call add(5, 3) → get 8
  2. Call multiply(2, 4) → get 8
  3. These could happen at the same time!

**Current problem:** Sequential execution is slow.

**Solution:** Use **fan-out/fan-in pattern** to:
1. Start multiple arithmetic operations simultaneously
2. Wait for all to complete
3. Combine results

## Theory Explanation

### What is Fan-out/Fan-in?
**Fan-out:** One node spawns multiple parallel branches
**Fan-in:** Multiple branches merge back to one node

```
        branch_1
       /         \
  Splitter  → branch_2 → Merger
       \         /
        branch_3
```

In arithmetic:
```
        add_node (5+3=8)
       /                  \
 LLM node → multiply_node (2*4=8) → Aggregator
       \                  /
        divide_node (10/2=5)
```

### How does LangGraph handle parallelism?
- LangGraph does NOT create async threads
- Instead, it **compiles** the graph to optimized structure
- At runtime, can execute branches in parallel (if using async)
- For deterministic arithmetic, parallel execution is safe

### Why Arithmetic is Safe to Parallelize
- Operations are **pure functions** (no side effects)
- Operations don't depend on each other
- Results are deterministic
- No race conditions

## What is New in This Lesson?

**In Lesson 8:** Sequential tool calls, one LLM loop

**In Lesson 9:**
- Multiple arithmetic nodes (add_node, multiply_node, divide_node)
- Conditional edges spawn parallel branches
- Aggregator node collects all results
- State stores results from all branches

## Which Imports / APIs Solve This Problem?

```python
# Multiple add_node() calls create parallel branches
graph.add_node("add_node", node_add)
graph.add_node("multiply_node", node_multiply)
graph.add_node("divide_node", node_divide)

# Conditional edges can route to multiple nodes at once (fan-out)
# To fan-in: multiple nodes route to same aggregator node
```

## Production Insight

In production:
1. Parallelization reduces latency (wall-clock time)
2. Multiple arithmetic operations often needed in enterprise workflows
3. Result aggregation must handle failures gracefully
4. Monitor single operations bottlenecks

## Full Working Code

### Setup

from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, List, Dict, Any

load_dotenv()
vertexai.init(
    project=os.getenv("PROJECT_ID"),
    location=os.getenv("LOCATION")
)

llm = ChatVertexAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("✓ Setup complete")

### Step 1: Define State with Parallel Results

class ParallelArithmeticState(TypedDict):
    """State for parallel arithmetic operations"""
    question: str                      # User question
    operands: List[tuple]              # List of (a, b) pairs to compute
    operations: List[str]              # Corresponding operations (add/multiply/divide)
    results: Dict[str, Any]            # Results keyed by operation index
    aggregated_result: Any             # Final aggregated result

print("✓ Defined ParallelArithmeticState")

### Step 2: Parse Node - LLM Interprets Question

def parse_node(state: ParallelArithmeticState) -> ParallelArithmeticState:
    """
    LLM parses user question to identify multiple parallel operations.
    
    Example input: "Add 5 and 3, multiply 2 and 4, divide 10 by 2"
    LLM extracts:
      operands: [(5, 3), (2, 4), (10, 2)]
      operations: ["add", "multiply", "divide"]
    """
    print(f"\n→ PARSE NODE:")
    print(f"  Question: {state['question']}")
    
    # For demo, hardcode parsing (in production, use LLM)
    # This is a simplified example
    operands = [(5, 3), (2, 4), (10, 2)]
    operations = ["add", "multiply", "divide"]
    
    print(f"  Parsed {len(operands)} operations to run in parallel:")
    for i, (op, (a, b)) in enumerate(zip(operations, operands)):
        print(f"    {i}: {op}({a}, {b})")
    
    return {
        "question": state["question"],
        "operands": operands,
        "operations": operations,
        "results": {},
        "aggregated_result": None
    }

print("✓ Defined parse_node")

### Step 3: Create Arithmetic Nodes (run in parallel)

def node_add(state: ParallelArithmeticState) -> ParallelArithmeticState:
    """
    Execute addition for operation index 0.
    Runs in parallel with multiply and divide nodes.
    """
    print(f"\n→ ADD NODE (parallel branch):")
    if len(state["operands"]) > 0:
        a, b = state["operands"][0]
        result = a + b
        print(f"  {a} + {b} = {result}")
        
        results = dict(state.get("results", {}))
        results["0_add"] = result
        
        return {
            "question": state["question"],
            "operands": state["operands"],
            "operations": state["operations"],
            "results": results,
            "aggregated_result": state["aggregated_result"]
        }
    return state

def node_multiply(state: ParallelArithmeticState) -> ParallelArithmeticState:
    """
    Execute multiplication for operation index 1.
    Runs in parallel with add and divide nodes.
    """
    print(f"\n→ MULTIPLY NODE (parallel branch):")
    if len(state["operands"]) > 1:
        a, b = state["operands"][1]
        result = a * b
        print(f"  {a} * {b} = {result}")
        
        results = dict(state.get("results", {}))
        results["1_multiply"] = result
        
        return {
            "question": state["question"],
            "operands": state["operands"],
            "operations": state["operations"],
            "results": results,
            "aggregated_result": state["aggregated_result"]
        }
    return state

def node_divide(state: ParallelArithmeticState) -> ParallelArithmeticState:
    """
    Execute division for operation index 2.
    Runs in parallel with add and multiply nodes.
    """
    print(f"\n→ DIVIDE NODE (parallel branch):")
    if len(state["operands"]) > 2:
        a, b = state["operands"][2]
        if b == 0:
            result = "Error: Division by zero"
        else:
            result = a // b
        print(f"  {a} / {b} = {result}")
        
        results = dict(state.get("results", {}))
        results["2_divide"] = result
        
        return {
            "question": state["question"],
            "operands": state["operands"],
            "operations": state["operations"],
            "results": results,
            "aggregated_result": state["aggregated_result"]
        }
    return state

print("✓ Defined add_node, multiply_node, divide_node")

### Step 4: Aggregator Node (Fan-in)

def aggregator_node(state: ParallelArithmeticState) -> ParallelArithmeticState:
    """
    Collects results from all parallel branches (fan-in).
    
    All parallel nodes have finished by this point.
    Results dictionary contains all arithmetic outputs.
    """
    print(f"\n→ AGGREGATOR NODE (fan-in):")
    print(f"  Collected results from parallel branches:")
    
    results = state.get("results", {})
    for key, value in results.items():
        print(f"    {key}: {value}")
    
    # Aggregate: for demo, just combine all results
    aggregated = {
        "operations_completed": len(results),
        "individual_results": results,
        "sum_of_results": sum(v for v in results.values() if isinstance(v, int))
    }
    
    return {
        "question": state["question"],
        "operands": state["operands"],
        "operations": state["operations"],
        "results": results,
        "aggregated_result": aggregated
    }

print("✓ Defined aggregator_node")

### Step 5: Build Graph with Fan-out/Fan-in

from langgraph.graph import StateGraph, START, END

graph = StateGraph(ParallelArithmeticState)

# Add nodes
graph.add_node("parse_node", parse_node)
graph.add_node("add_node", node_add)           # Parallel branch 1
graph.add_node("multiply_node", node_multiply) # Parallel branch 2
graph.add_node("divide_node", node_divide)     # Parallel branch 3
graph.add_node("aggregator", aggregator_node)

# Start to parse
graph.add_edge(START, "parse_node")

# Fan-out: parse routes to all three parallel nodes
graph.add_edge("parse_node", "add_node")
graph.add_edge("parse_node", "multiply_node")
graph.add_edge("parse_node", "divide_node")

# Fan-in: all parallel nodes route to aggregator
graph.add_edge("add_node", "aggregator")
graph.add_edge("multiply_node", "aggregator")
graph.add_edge("divide_node", "aggregator")

# End
graph.add_edge("aggregator", END)

parallel_arithmetic_graph = graph.compile()
print("✓ Compiled graph with fan-out/fan-in")

### Step 6: Visualize the Parallel Graph

from IPython.display import Image, display

graph_image = parallel_arithmetic_graph.get_graph().draw_mermaid_png()
display(Image(graph_image))
print("✓ Graph visualization shows fan-out and fan-in structure")

### Step 7: Run Tests

initial_state = {
    "question": "Compute 5+3, 2*4, and 10/2 in parallel",
    "operands": [],
    "operations": [],
    "results": {},
    "aggregated_result": None
}

print("\n" + "="*60)
print("Running Parallel Arithmetic Graph")
print("="*60)

final_state = parallel_arithmetic_graph.invoke(initial_state)

print(f"\n{'='*60}")
print("FINAL RESULTS")
print(f"{'='*60}")
print(f"Results: {final_state['results']}")
print(f"Aggregated: {final_state['aggregated_result']}")

## Step-by-Step Code Explanation

### 1. **Multiple add_node() Calls**
```python
graph.add_node("add_node", node_add)
graph.add_node("multiply_node", node_multiply)
graph.add_node("divide_node", node_divide)
```
**Why?**
- Each node is a separate operation
- All three can execute without dependencies on each other
- Graph compiler optimizes to run them in parallel

### 2. **Fan-out: One node → Multiple nodes**
```python
graph.add_edge("parse_node", "add_node")
graph.add_edge("parse_node", "multiply_node")
graph.add_edge("parse_node", "divide_node")
```
**Why?**
- Same source node connecting to multiple targets
- Graph knows all three can proceed independently
- Execution engine can parallelize

### 3. **Fan-in: Multiple nodes → One node**
```python
graph.add_edge("add_node", "aggregator")
graph.add_edge("multiply_node", "aggregator")
graph.add_edge("divide_node", "aggregator")
```
**Why?**
- Aggregator waits for ALL parallel nodes to finish
- Implicit synchronization point
- Collects results from all branches

### 4. **Result Merging**
```python
results = dict(state.get("results", {}))
results["0_add"] = result  # Each node updates same dict
```
**Why?**
- State is immutable, but we create new dict
- Each parallel node merges its result into dict
- Aggregator sees all results combined

## Summary

**What changed from Lesson 8?**
1. Lesson 8: Sequential LLM loops, one tool at a time
2. Lesson 9: Parallel arithmetic, all operations at once

**Architecture:**
```
parse_node
   ↓ → add_node
   ↓    ↓
   ↓ → multiply_node → aggregator
   ↓    ↓
   ↓ → divide_node
```

## Why This Matters in Production

1. **Latency**: Parallel execution reduces wall-clock time
   - Sequential: 3 operations × 100ms = 300ms
   - Parallel: max(3 × 100ms) = 100ms

2. **Real-world example**: Compute multiple financial metrics simultaneously
   - Revenue, cost, profit could run in parallel
   - Results aggregated for dashboard

3. **Failure handling**: If one branch fails (divide by zero), others continue
   - Aggregator can handle partial results

**Next lesson:** We'll apply map-reduce pattern to lists of numbers.